> **Versión GitHub reproducible.** Esta copia no modifica el notebook original de Google Drive. Las rutas personales de Drive se han sustituido por rutas del repositorio. Los datos brutos de gran tamaño no incluidos en GitHub deben descargarse según `docs/DATA_SOURCES.md`.


In [ ]:
# ============================================================
# CONFIGURACIÓN DE RUTAS - VERSIÓN GITHUB / GOOGLE COLAB
# ============================================================
# Clone primero el repositorio con:
# !git clone https://github.com/ritaridh/tfm-crop-yield-prediction.git /content/TFM_Rendimiento_cultivos
# %cd /content/TFM_Rendimiento_cultivos

from pathlib import Path
import os

ROOT = Path("/content/TFM_Rendimiento_cultivos")
if not ROOT.exists():
    raise FileNotFoundError(
        "No se encuentra el repositorio. Ejecute primero el bloque de clonación indicado en el README."
    )

RUTA_TFM = str(ROOT)
RUTA_BASE = str(ROOT)
BASE = str(ROOT)
BASE_DIR = ROOT
RUTA_ORIGINALES = str(ROOT / "01_Datos_originales")
RUTA_EUROCROPS = str(ROOT / "01_Datos_originales" / "EuroCrops")
RUTA_MAPA = str(ROOT / "01_Datos_originales" / "MAPA")
RUTA_SENTINEL = str(ROOT / "01_Datos_originales" / "Sentinel2")
RUTA_PROCESADOS = str(ROOT / "02_Datos_procesados")
RUTA_RESULTADOS = str(ROOT / "05_Resultados")
PROC = RUTA_PROCESADOS
RESULTADOS = RUTA_RESULTADOS
Path(RUTA_PROCESADOS).mkdir(parents=True, exist_ok=True)
Path(RUTA_RESULTADOS).mkdir(parents=True, exist_ok=True)
print("Repositorio:", ROOT)


In [ ]:
#-------------------------------
# Montar drive
#------------------------------


In [ ]:
#-------------------------------
# Definir rutas permanentes
#-------------------------------

import os
import pandas as pd
import numpy as np

# ============================================================
# RUTAS PERMANENTES DEL TFM EN GOOGLE DRIVE
# ============================================================

RUTA_BASE = "/content/TFM_Rendimiento_cultivos"

RUTA_ORIGINALES = os.path.join(
    RUTA_BASE,
    "01_Datos_originales"
)

RUTA_EUROCROPS = os.path.join(
    RUTA_ORIGINALES,
    "EuroCrops"
)

RUTA_MAPA = os.path.join(
    RUTA_ORIGINALES,
    "MAPA"
)

RUTA_PROCESADOS = os.path.join(
    RUTA_BASE,
    "02_Datos_procesados"
)

RUTA_RESULTADOS = os.path.join(
    RUTA_BASE,
    "05_Resultados"
)

print("RUTA BASE:", RUTA_BASE)
print("Procesados:", RUTA_PROCESADOS)
print("Resultados:", RUTA_RESULTADOS)

In [ ]:
# ============================================================
# CARGAR TARGET DE RENDIMIENTO
# ============================================================

ruta_target = os.path.join(
    RUTA_PROCESADOS,
    "target_trigo_cebada_2017_2023.csv"
)

target = pd.read_csv(ruta_target)

print("Archivo cargado:")
print(ruta_target)

print("\nDimensiones:")
print(target.shape)

print("\nColumnas:")
print(target.columns.tolist())

print("\nPrimeras filas:")
display(target.head())

In [ ]:
#---------------------------
# Comprobación antes de ERA5
#---------------------------

print("Observaciones:", len(target))

print("\nObservaciones por cultivo:")
display(
    target.groupby("cultivo")
          .size()
          .to_frame("n_observaciones")
)

print("\nAños disponibles:")
print(sorted(target["ano"].unique()))

print("\nValores nulos:")
display(target.isna().sum())

# Configuración de Copernicus Climate Data Store (CDS)

In [ ]:
!pip -q install cdsapi

In [ ]:
import cdsapi

print("cdsapi instalado correctamente")

# He creado key secret con API de Climate Data Store (ERA5)

In [ ]:
# Comprobar que Colab puede leer)

from google.colab import userdata

CDS_API_KEY = userdata.get("CDS_API_KEY")

print("API key disponible:", CDS_API_KEY is not None)

In [ ]:
#------------------------------
# Crear cliente de Copernicus
#-------------------------------

import cdsapi
from google.colab import userdata

CDS_API_KEY = userdata.get("CDS_API_KEY")

cds = cdsapi.Client(
    url="https://cds.climate.copernicus.eu/api",
    key=CDS_API_KEY
)

print("Cliente CDS configurado correctamente.")

In [ ]:
#--------------------------------------
# Crear carpeta permanente para ERA 5 en Drive
#--------------------------------------

import os

RUTA_ERA5 = os.path.join(
    RUTA_ORIGINALES,
    "ERA5_Land"
)

os.makedirs(RUTA_ERA5, exist_ok=True)

print("Carpeta ERA5-Land:")
print(RUTA_ERA5)
print("Existe:", os.path.exists(RUTA_ERA5))

# Prueba 3 días ERA5-Land hourly data from 1950 to present

In [ ]:
#--------------------------
# Descargar prueba de 1 al 3 de enero 2023
#---------------------------

import os

archivo_prueba = os.path.join(
    RUTA_ERA5,
    "ERA5Land_prueba_2023_01_01_03.nc"
)

request = {
    "variable": [
        "2m_temperature",
        "2m_dewpoint_temperature",
        "total_precipitation",
        "surface_solar_radiation_downwards"
    ],
    "year": "2023",
    "month": "01",
    "day": [
        "01", "02", "03"
    ],
    "time": [
        "00:00", "01:00", "02:00", "03:00",
        "04:00", "05:00", "06:00", "07:00",
        "08:00", "09:00", "10:00", "11:00",
        "12:00", "13:00", "14:00", "15:00",
        "16:00", "17:00", "18:00", "19:00",
        "20:00", "21:00", "22:00", "23:00"
    ],

    # Norte, Oeste, Sur, Este
    # Península + Baleares aproximadamente
    "area": [
        44.5, -10.0,
        35.0, 4.5
    ],

    "data_format": "netcdf",
    "download_format": "unarchived"
}

cds.retrieve(
    "reanalysis-era5-land",
    request,
    archivo_prueba
)

print("Archivo guardado en:")
print(archivo_prueba)

print(
    "Existe:",
    os.path.exists(archivo_prueba)
)

#Inspeccionar archivo de prueba

In [ ]:
# Abrir archivo en Xarray

import xarray as xr
import os

archivo_prueba = os.path.join(
    RUTA_ERA5,
    "ERA5Land_prueba_2023_01_01_03.nc"
)

ds = xr.open_dataset(archivo_prueba)

print(ds)

In [ ]:
# Ver variables y unidades

print("VARIABLES ERA5-LAND\n")

for variable in ds.data_vars:
    print("=" * 60)
    print("Variable:", variable)
    print("Nombre:", ds[variable].attrs.get("long_name"))
    print("Unidades:", ds[variable].attrs.get("units"))
    print("Dimensiones:", ds[variable].dims)
    print("Shape:", ds[variable].shape)

In [ ]:
# Comprobar coordenadas y periodo temporal

print("COORDENADAS:")
print(list(ds.coords))

print("\nRango temporal:")

if "valid_time" in ds.coords:
    print("Inicio:", ds["valid_time"].values[0])
    print("Fin:", ds["valid_time"].values[-1])
    print("Nº observaciones temporales:", ds.sizes["valid_time"])

elif "time" in ds.coords:
    print("Inicio:", ds["time"].values[0])
    print("Fin:", ds["time"].values[-1])
    print("Nº observaciones temporales:", ds.sizes["time"])

# Prueba 1 mes ERA5-Land monthly averaged data from 1950 to present

In [ ]:
import os

archivo_prueba_mensual = os.path.join(
    RUTA_ERA5,
    "ERA5Land_mensual_campana_2023.nc"
)

request_mensual = {
    "product_type": [
        "monthly_averaged_reanalysis"
    ],

    "variable": [
        "2m_temperature",
        "2m_dewpoint_temperature",
        "total_precipitation",
        "surface_solar_radiation_downwards"
    ],

    "year": [
        "2022",
        "2023"
    ],

    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "10", "11", "12"
    ],

    "time": [
        "00:00"
    ],

    "data_format": "netcdf",

    "download_format": "unarchived",

    # España peninsular + Baleares
    "area": [
        44.5, -10.0,
        35.0, 4.5
    ]
}

cds.retrieve(
    "reanalysis-era5-land-monthly-means",
    request_mensual,
    archivo_prueba_mensual
)

print("Archivo guardado:")
print(archivo_prueba_mensual)

print("Existe:", os.path.exists(archivo_prueba_mensual))

# Comprobrar estructura del NetCDF mensual

In [ ]:
import xarray as xr
import os

archivo_mensual = os.path.join(
    RUTA_ERA5,
    "ERA5Land_mensual_campana_2023.nc"
)

ds_mensual = xr.open_dataset(archivo_mensual)

print(ds_mensual)

In [ ]:
print("VARIABLES ERA5-LAND MENSUAL\n")

for variable in ds_mensual.data_vars:
    print("=" * 60)
    print("Variable:", variable)
    print("Nombre:", ds_mensual[variable].attrs.get("long_name"))
    print("Unidades:", ds_mensual[variable].attrs.get("units"))
    print("Dimensiones:", ds_mensual[variable].dims)
    print("Shape:", ds_mensual[variable].shape)

In [ ]:
print("COORDENADAS:")
print(list(ds_mensual.coords))

if "valid_time" in ds_mensual.coords:
    print("\nFechas disponibles:")
    print(ds_mensual["valid_time"].values)

elif "time" in ds_mensual.coords:
    print("\nFechas disponibles:")
    print(ds_mensual["time"].values)

# Descargar ERA5-Land mensual 2016–2023

In [ ]:
import os
import cdsapi

os.makedirs(RUTA_ERA5, exist_ok=True)

variables_era5 = [
    "2m_temperature",
    "2m_dewpoint_temperature",
    "total_precipitation",
    "surface_solar_radiation_downwards"
]

# Ventana meteorológica:
# septiembre del año anterior -> agosto del año de cosecha

meses_por_ano = {
    2016: ["09", "10", "11", "12"],

    2017: ["01", "02", "03", "04", "05", "06", "07", "08",
           "09", "10", "11", "12"],

    2018: ["01", "02", "03", "04", "05", "06", "07", "08",
           "09", "10", "11", "12"],

    2019: ["01", "02", "03", "04", "05", "06", "07", "08",
           "09", "10", "11", "12"],

    2020: ["01", "02", "03", "04", "05", "06", "07", "08",
           "09", "10", "11", "12"],

    2021: ["01", "02", "03", "04", "05", "06", "07", "08",
           "09", "10", "11", "12"],

    2022: ["01", "02", "03", "04", "05", "06", "07", "08",
           "09", "10", "11", "12"],

    2023: ["01", "02", "03", "04", "05", "06", "07", "08"]
}

n_meses = sum(len(meses) for meses in meses_por_ano.values())

print("Meses únicos que se descargarán:", n_meses)

# Descargar los 8 archivos anuales

In [ ]:
for ano, meses in meses_por_ano.items():

    archivo_salida = os.path.join(
        RUTA_ERA5,
        f"ERA5Land_mensual_{ano}_con_agosto.nc"
    )

    if os.path.exists(archivo_salida):
        print(f"{ano}: versión con agosto ya existe -> se omite")
        continue

    print(f"\nDescargando {ano}...")
    print("Meses:", meses)

    request = {
        "product_type": ["monthly_averaged_reanalysis"],

        "variable": variables_era5,

        "year": [str(ano)],

        "month": meses,

        "time": ["00:00"],

        "data_format": "netcdf",

        "download_format": "unarchived",

        "area": [
            44.5, -10.0,
            35.0, 4.5
        ]
    }

    cds.retrieve(
        "reanalysis-era5-land-monthly-means",
        request,
        archivo_salida
    )

    print(f"{ano}: OK")
    print("Guardado en:", archivo_salida)

print("\nDescarga con agosto terminada.")

In [ ]:
import os
import xarray as xr
import pandas as pd

# ============================================================
# COMPROBAR DESCARGA ERA5-LAND CON AGOSTO
# 2016-2023
# ============================================================

resumen_era5 = []

for ano in range(2016, 2024):

    ruta = os.path.join(
        RUTA_ERA5,
        f"ERA5Land_mensual_{ano}_con_agosto.nc"
    )

    if not os.path.exists(ruta):
        resumen_era5.append({
            "ano": ano,
            "archivo_existe": False,
            "n_meses": 0
        })
        continue

    ds_tmp = xr.open_dataset(ruta)

    n_meses = ds_tmp.sizes["valid_time"]

    resumen_era5.append({
        "ano": ano,
        "archivo_existe": True,
        "n_meses": n_meses,
        "fecha_inicio": pd.to_datetime(
            ds_tmp["valid_time"].values[0]
        ).strftime("%Y-%m"),
        "fecha_fin": pd.to_datetime(
            ds_tmp["valid_time"].values[-1]
        ).strftime("%Y-%m")
    })

    ds_tmp.close()

resumen_era5_df = pd.DataFrame(resumen_era5)

display(resumen_era5_df)

print(
    "\nTotal de meses descargados:",
    resumen_era5_df["n_meses"].sum()
)

# Descargar ERAS5-Land mensual para España

In [ ]:
# ============================================================
# ERA5-LAND MENSUAL DEFINITIVO - ESPAÑA COMPLETA
# Península + Baleares + Canarias
# Septiembre 2016 - Agosto 2023
# ============================================================

for ano, meses in meses_por_ano.items():

    archivo_salida = os.path.join(
        RUTA_ERA5,
        f"ERA5Land_mensual_{ano}_espana.nc"
    )

    # No repetir descargas ya completadas
    if os.path.exists(archivo_salida):
        print(f"{ano}: España completa ya existe -> se omite")
        continue

    print(f"\nDescargando España completa {ano}...")
    print("Meses:", meses)

    request = {
        "product_type": [
            "monthly_averaged_reanalysis"
        ],

        "variable": variables_era5,

        "year": [
            str(ano)
        ],

        "month": meses,

        "time": [
            "00:00"
        ],

        "data_format": "netcdf",

        "download_format": "unarchived",

        # Norte, Oeste, Sur, Este
        # Península + Baleares + Canarias
        "area": [
            44.5, -19.0,
            27.0, 4.5
        ]
    }

    cds.retrieve(
        "reanalysis-era5-land-monthly-means",
        request,
        archivo_salida
    )

    print(f"{ano}: OK")
    print("Guardado en:", archivo_salida)

print("\nDescarga definitiva de España terminada.")


In [ ]:
# ============================================================
# COMPROBAR ERA5-LAND DEFINITIVO ESPAÑA
# ============================================================

import xarray as xr
import pandas as pd
import os

resumen_era5_espana = []

for ano in range(2016, 2024):

    ruta = os.path.join(
        RUTA_ERA5,
        f"ERA5Land_mensual_{ano}_espana.nc"
    )

    if not os.path.exists(ruta):
        resumen_era5_espana.append({
            "ano": ano,
            "archivo_existe": False,
            "n_meses": 0
        })
        continue

    ds_tmp = xr.open_dataset(ruta)

    resumen_era5_espana.append({
        "ano": ano,
        "archivo_existe": True,
        "n_meses": ds_tmp.sizes["valid_time"],
        "fecha_inicio": pd.to_datetime(
            ds_tmp["valid_time"].values[0]
        ).strftime("%Y-%m"),
        "fecha_fin": pd.to_datetime(
            ds_tmp["valid_time"].values[-1]
        ).strftime("%Y-%m")
    })

    ds_tmp.close()

resumen_era5_espana_df = pd.DataFrame(
    resumen_era5_espana
)

display(resumen_era5_espana_df)

print(
    "\nTOTAL MESES DEFINITIVOS:",
    resumen_era5_espana_df["n_meses"].sum()
)

#Unir los 8 NetCDF definitivos de España en un solo xarray.Dataset temporal

In [ ]:
import os
import xarray as xr
import pandas as pd

# ============================================================
# UNIR LOS 8 NETCDF DEFINITIVOS DE ERA5-LAND
# ============================================================

archivos_era5_espana = [
    os.path.join(
        RUTA_ERA5,
        f"ERA5Land_mensual_{ano}_espana.nc"
    )
    for ano in range(2016, 2024)
]

ds_era5 = xr.open_mfdataset(
    archivos_era5_espana,
    combine="by_coords"
)

print(ds_era5)

print("\nNúmero total de meses:")
print(ds_era5.sizes["valid_time"])

print("\nInicio:")
print(pd.to_datetime(ds_era5["valid_time"].values[0]))

print("\nFin:")
print(pd.to_datetime(ds_era5["valid_time"].values[-1]))

In [ ]:
# ============================================================
# ASIGNAR CADA MES A SU AÑO DE COSECHA / CAMPAÑA
# ============================================================

fechas = pd.to_datetime(ds_era5["valid_time"].values)

tabla_fechas = pd.DataFrame({
    "fecha": fechas
})

tabla_fechas["ano_calendario"] = tabla_fechas["fecha"].dt.year
tabla_fechas["mes"] = tabla_fechas["fecha"].dt.month

tabla_fechas["ano_campana"] = tabla_fechas.apply(
    lambda fila:
        fila["ano_calendario"] + 1
        if fila["mes"] >= 9
        else fila["ano_calendario"],
    axis=1
)

display(tabla_fechas)

In [ ]:
print("Meses por campaña:")

display(
    tabla_fechas
    .groupby("ano_campana")
    .size()
    .to_frame("n_meses")
)

# Transformación de variables meteorológicas

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np

# ============================================================
# CONVERSIÓN DE VARIABLES ERA5-LAND
# Dataset original intacto: ds_era5
# ============================================================

# Número de días de cada mes
fechas = pd.to_datetime(ds_era5["valid_time"].values)

dias_mes = xr.DataArray(
    fechas.days_in_month,
    coords={"valid_time": ds_era5["valid_time"]},
    dims=["valid_time"]
)

ds_meteo = xr.Dataset(
    {
        # Temperatura media mensual: Kelvin -> ºC
        "temp_media_c": (
            ds_era5["t2m"] - 273.15
        ),

        # Temperatura de rocío media mensual: Kelvin -> ºC
        "temp_rocio_c": (
            ds_era5["d2m"] - 273.15
        ),

        # Precipitación:
        # m/día -> mm/mes
        "precipitacion_mm": (
            ds_era5["tp"] * 1000 * dias_mes
        ),

        # Radiación:
        # J/m²/día -> MJ/m²/mes
        "radiacion_mj_m2": (
            ds_era5["ssrd"] * dias_mes / 1_000_000
        )
    }
)

print(ds_meteo)

In [ ]:
# ============================================================
# AÑADIR MES Y AÑO DE CAMPAÑA
# Septiembre-diciembre -> cosecha año siguiente
# Enero-agosto -> cosecha mismo año
# ============================================================

fechas = pd.to_datetime(
    ds_meteo["valid_time"].values
)

mes = fechas.month

ano_campana = np.where(
    mes >= 9,
    fechas.year + 1,
    fechas.year
)

ds_meteo = ds_meteo.assign_coords(
    mes=("valid_time", mes),
    ano_campana=("valid_time", ano_campana)
)

print("Campañas disponibles:")
print(np.unique(ds_meteo["ano_campana"].values))

print("\nMeses disponibles por campaña:")

for campana in range(2017, 2024):

    meses = ds_meteo["mes"].where(
        ds_meteo["ano_campana"] == campana,
        drop=True
    ).values

    print(
        campana,
        "->",
        list(meses),
        "| n =", len(meses)
    )

In [ ]:
# ============================================================
# CONTROL DE RANGOS DESPUÉS DE CONVERTIR UNIDADES
# ============================================================

for variable in [
    "temp_media_c",
    "temp_rocio_c",
    "precipitacion_mm",
    "radiacion_mj_m2"
]:

    print("=" * 60)
    print(variable)

    print(
        "mínimo:",
        float(ds_meteo[variable].min().values)
    )

    print(
        "máximo:",
        float(ds_meteo[variable].max().values)
    )

    print(
        "media:",
        float(ds_meteo[variable].mean().values)
    )

# Provincias - límites geográficos de las provincias españolas para calcular, para cada provincia, la media de las celdas ERA5-Land que caen dentro de ella.

In [ ]:
import os

print("Archivos disponibles en 02_Datos_procesados:\n")

for archivo in sorted(os.listdir(RUTA_PROCESADOS)):
    print(archivo)

In [ ]:
import os

print("Contenido de EuroCrops:\n")

for raiz, carpetas, archivos in os.walk(RUTA_EUROCROPS):

    nivel = raiz.replace(RUTA_EUROCROPS, "").count(os.sep)

    # Para no imprimir una cantidad enorme de información
    if nivel <= 2:
        indentacion = "    " * nivel
        print(f"{indentacion}{os.path.basename(raiz)}/")

        for archivo in archivos[:10]:
            print(f"{indentacion}    {archivo}")

        if len(archivos) > 10:
            print(
                f"{indentacion}    ... "
                f"({len(archivos)} archivos en total)"
            )

In [ ]:
# Crear carpeta en DRIVE para Limites de rutas con CNIG/IGN

import os

RUTA_LIMITES = os.path.join(
    RUTA_ORIGINALES,
    "Limites_administrativos"
)

os.makedirs(RUTA_LIMITES, exist_ok=True)

print("Ruta límites:")
print(RUTA_LIMITES)

print("Existe:", os.path.exists(RUTA_LIMITES))

# Descargo datos de https://centrodedescargas.cnig.es/CentroDescargas/limites-municipales-provinciales-autonomicos

In [ ]:
import os

RUTA_LIMITES = os.path.join(
    RUTA_ORIGINALES,
    "Limites_administrativos"
)

print("Contenido de Limites_administrativos:\n")

for raiz, carpetas, archivos in os.walk(RUTA_LIMITES):
    nivel = raiz.replace(RUTA_LIMITES, "").count(os.sep)
    indent = "    " * nivel

    print(f"{indent}{os.path.basename(raiz)}/")

    for archivo in archivos:
        print(f"{indent}    {archivo}")

In [ ]:
!pip -q install geopandas

In [ ]:
import geopandas as gpd
import pandas as pd
import os

print("GeoPandas:", gpd.__version__)

In [ ]:
# ============================================================
# RUTAS DE LOS RECINTOS PROVINCIALES OFICIALES CNIG
# ============================================================

ruta_provincias_peninsula = os.path.join(
    RUTA_LIMITES,
    "lineas_limite",
    "SHP_ETRS89",
    "recintos_provinciales_inspire_peninbal_etrs89",
    "recintos_provinciales_inspire_peninbal_etrs89.shp"
)

ruta_provincias_canarias = os.path.join(
    RUTA_LIMITES,
    "lineas_limite",
    "SHP_WGS84",
    "recintos_provinciales_inspire_canarias_wgs84",
    "recintos_provinciales_inspire_canarias_wgs84.shp"
)

print("Península+Baleares:", os.path.exists(ruta_provincias_peninsula))
print("Canarias:", os.path.exists(ruta_provincias_canarias))

In [ ]:
provincias_peninsula = gpd.read_file(
    ruta_provincias_peninsula
)

provincias_canarias = gpd.read_file(
    ruta_provincias_canarias
)

print("Península+Baleares:")
print(provincias_peninsula.shape)
print("CRS:", provincias_peninsula.crs)

print("\nCanarias:")
print(provincias_canarias.shape)
print("CRS:", provincias_canarias.crs)

In [ ]:
print("COLUMNAS PENÍNSULA + BALEARES:")
print(provincias_peninsula.columns.tolist())

print("\nPrimeras filas:")
display(provincias_peninsula.head())

print("\nCOLUMNAS CANARIAS:")
print(provincias_canarias.columns.tolist())

print("\nPrimeras filas:")
display(provincias_canarias.head())

In [ ]:
# ============================================================
# UNIR RECINTOS PROVINCIALES CNIG
# CRS comun: EPSG:4326
# ============================================================

import geopandas as gpd
import pandas as pd

# Peninsula + Baleares: ETRS89 -> WGS84
provincias_peninsula_4326 = provincias_peninsula.to_crs(epsg=4326)

# Canarias ya esta en EPSG:4326
provincias_canarias_4326 = provincias_canarias.to_crs(epsg=4326)

# Seleccionar nombre y geometria
prov_pen = provincias_peninsula_4326[
    ["NAMEUNIT", "geometry"]
].copy()

prov_can = provincias_canarias_4326[
    ["NAMEUNIT", "geometry"]
].copy()

# Unir ambos conjuntos
provincias_cnig = gpd.GeoDataFrame(
    pd.concat(
        [prov_pen, prov_can],
        ignore_index=True
    ),
    crs="EPSG:4326"
)

print("Registros CNIG:", len(provincias_cnig))
print("CRS:", provincias_cnig.crs)

In [ ]:
# ============================================================
# COMPARAR NOMBRES CNIG VS TARGET MAPA
# ============================================================

nombres_cnig = sorted(
    provincias_cnig["NAMEUNIT"]
    .dropna()
    .unique()
)

provincias_target = sorted(
    target["provincia_estandar"]
    .dropna()
    .unique()
)

print("Número de nombres CNIG:", len(nombres_cnig))
print("Número de provincias target:", len(provincias_target))

print("\nNOMBRES CNIG:")
for nombre in nombres_cnig:
    print(repr(nombre))

print("\nPROVINCIAS TARGET MAPA:")
for nombre in provincias_target:
    print(repr(nombre))

print("\nCNIG que no coinciden exactamente con MAPA:")
for nombre in sorted(set(nombres_cnig) - set(provincias_target)):
    print(repr(nombre))

print("\nMAPA que no coinciden exactamente con CNIG:")
for nombre in sorted(set(provincias_target) - set(nombres_cnig)):
    print(repr(nombre))

In [ ]:
# ============================================================
# ARMONIZAR PROVINCIAS CNIG CON MAPA
# ============================================================

# Copia de seguridad
provincias_final = provincias_cnig.copy()

# 1. Excluir unidades que no forman parte de las 50 provincias MAPA
excluir = [
    "Ceuta",
    "Melilla",
    "Territorios no asociados a ninguna provincia"
]

provincias_final = provincias_final[
    ~provincias_final["NAMEUNIT"].isin(excluir)
].copy()

# 2. Armonizar las diferencias de nomenclatura CNIG -> MAPA
equivalencias = {
    "Alacant/Alicante": "Alicante",
    "Araba/Álava": "Álava",
    "Castelló/Castellón": "Castellón",
    "València/Valencia": "Valencia"
}

provincias_final["provincia_estandar"] = (
    provincias_final["NAMEUNIT"]
    .replace(equivalencias)
)

# 3. Comprobaciones
print("Número de geometrías:", len(provincias_final))
print(
    "Número de provincias únicas:",
    provincias_final["provincia_estandar"].nunique()
)

print(
    "CRS:",
    provincias_final.crs
)

print(
    "Geometrías nulas:",
    provincias_final.geometry.isna().sum()
)

print(
    "Geometrías inválidas:",
    (~provincias_final.geometry.is_valid).sum()
)

In [ ]:
# ============================================================
# COMPROBAR CORRESPONDENCIA EXACTA CNIG <-> MAPA
# ============================================================

set_cnig = set(
    provincias_final["provincia_estandar"]
)

set_mapa = set(
    target["provincia_estandar"].dropna().unique()
)

print("Provincias CNIG:", len(set_cnig))
print("Provincias MAPA:", len(set_mapa))

print(
    "\nEn CNIG pero no en MAPA:",
    sorted(set_cnig - set_mapa)
)

print(
    "En MAPA pero no en CNIG:",
    sorted(set_mapa - set_cnig)
)

print(
    "\nCorrespondencia exacta:",
    set_cnig == set_mapa
)

In [ ]:
# ============================================================
# COMPROBAR BIZKAIA EN LOS DATOS MAPA
# ============================================================

import pandas as pd
import os

ruta_mapa_procesado = os.path.join(
    RUTA_PROCESADOS,
    "rendimientos_MAPA_trigo_cebada_2017_2023.csv"
)

mapa_completo = pd.read_csv(ruta_mapa_procesado)

print("Columnas:")
print(mapa_completo.columns.tolist())

print("\nDimensiones:")
print(mapa_completo.shape)

print("\nRegistros relacionados con Bizkaia/Vizcaya:")

mask = mapa_completo.astype(str).apply(
    lambda col: col.str.contains(
        "Bizkaia|Vizcaya",
        case=False,
        na=False
    )
).any(axis=1)

display(mapa_completo[mask])

In [ ]:
# ============================================================
# GUARDAR PROVINCIAS CNIG PROCESADAS EN DRIVE
# ============================================================

ruta_provincias_procesadas = os.path.join(
    RUTA_PROCESADOS,
    "provincias_CNIG_50.gpkg"
)

# Guardar únicamente las columnas necesarias
provincias_guardar = provincias_final[
    ["provincia_estandar", "geometry"]
].copy()

provincias_guardar.to_file(
    ruta_provincias_procesadas,
    layer="provincias",
    driver="GPKG"
)

print("Archivo guardado:")
print(ruta_provincias_procesadas)

print("\nExiste:", os.path.exists(ruta_provincias_procesadas))
print("Provincias:", len(provincias_guardar))
print("CRS:", provincias_guardar.crs)

In [ ]:
provincias_final = gpd.read_file(
    os.path.join(
        RUTA_PROCESADOS,
        "provincias_CNIG_50.gpkg"
    )
)

# Agregación espacial de ERA5-Land por provincia

In [ ]:
# ============================================================
# COMPROBAR MALLA ERA5-LAND FRENTE A LAS PROVINCIAS
# ============================================================

import xarray as xr
import os
import numpy as np
import pandas as pd
import geopandas as gpd

# Usamos 2022 solo para inspeccionar la estructura espacial
ruta_era5_prueba = os.path.join(
    RUTA_ERA5,
    "ERA5Land_mensual_2022_con_agosto.nc"
)

print("Archivo existe:", os.path.exists(ruta_era5_prueba))

ds = xr.open_dataset(ruta_era5_prueba)

print("\nDimensiones:")
print(ds.sizes)

print("\nLatitudes:")
print("n =", len(ds.latitude))
print("mínimo =", float(ds.latitude.min()))
print("máximo =", float(ds.latitude.max()))

print("\nLongitudes:")
print("n =", len(ds.longitude))
print("mínimo =", float(ds.longitude.min()))
print("máximo =", float(ds.longitude.max()))

# Resolución aproximada de la malla
res_lat = abs(
    float(ds.latitude.values[1] - ds.latitude.values[0])
)

res_lon = abs(
    float(ds.longitude.values[1] - ds.longitude.values[0])
)

print("\nResolución espacial:")
print("Latitud:", res_lat, "grados")
print("Longitud:", res_lon, "grados")

print("\nVariables:")
print(list(ds.data_vars))

In [ ]:
# ============================================================
# CREAR PUNTOS DE LA MALLA ERA5-LAND
# ============================================================

from shapely.geometry import Point
import geopandas as gpd
import pandas as pd
import numpy as np

# Coordenadas de ERA5
latitudes = ds["latitude"].values
longitudes = ds["longitude"].values

# Crear todas las combinaciones lon-lat
lon_grid, lat_grid = np.meshgrid(
    longitudes,
    latitudes
)

# Convertir a tabla
puntos_era5 = pd.DataFrame({
    "longitude": lon_grid.ravel(),
    "latitude": lat_grid.ravel()
})

# Convertir a GeoDataFrame
puntos_era5 = gpd.GeoDataFrame(
    puntos_era5,
    geometry=gpd.points_from_xy(
        puntos_era5["longitude"],
        puntos_era5["latitude"]
    ),
    crs="EPSG:4326"
)

print("Número total de puntos ERA5:")
print(len(puntos_era5))

print("\nCRS:")
print(puntos_era5.crs)

display(puntos_era5.head())

In [ ]:
# ============================================================
# ASIGNAR PUNTOS ERA5 A PROVINCIAS
# ============================================================

puntos_provincia = gpd.sjoin(
    puntos_era5,
    provincias_final[
        ["provincia_estandar", "geometry"]
    ],
    how="inner",
    predicate="within"
)

print("Puntos ERA5 dentro de provincias españolas:")
print(len(puntos_provincia))

print("\nNúmero de provincias representadas:")
print(
    puntos_provincia["provincia_estandar"].nunique()
)

# Número de puntos ERA5 por provincia
conteo_puntos = (
    puntos_provincia
    .groupby("provincia_estandar")
    .size()
    .sort_values()
    .to_frame("n_puntos_ERA5")
)

display(conteo_puntos)

In [ ]:
# ============================================================
# COMPROBAR PROVINCIAS SIN PUNTOS ERA5-LAND
# ============================================================

provincias_cnig = set(
    provincias_final["provincia_estandar"].unique()
)

provincias_era5 = set(
    puntos_provincia["provincia_estandar"].unique()
)

sin_era5 = sorted(
    provincias_cnig - provincias_era5
)

print("Provincias CNIG:", len(provincias_cnig))
print("Provincias con puntos ERA5:", len(provincias_era5))

print("\nProvincias sin puntos ERA5:")
for provincia in sin_era5:
    print("-", provincia)

In [ ]:
# ============================================================
# COMPROBAR SI LAS PROVINCIAS SIN ERA5
# TIENEN OBSERVACIONES DE RENDIMIENTO EN EL TARGET
# ============================================================

target_sin_era5 = target[
    target["provincia_estandar"].isin(sin_era5)
].copy()

print("Observaciones del target en provincias sin ERA5:")
print(len(target_sin_era5))

if len(target_sin_era5) > 0:
    display(
        target_sin_era5.sort_values(
            ["provincia_estandar", "cultivo", "ano"]
        )
    )
else:
    print(
        "Ninguna provincia sin ERA5 tiene "
        "observaciones válidas en el target."
    )

In [ ]:
# ============================================================
# CONFIGURAR DESCARGA ERA5-LAND PARA CANARIAS
# ============================================================

import os

RUTA_ERA5_CANARIAS = os.path.join(
    RUTA_ERA5,
    "Canarias"
)

os.makedirs(
    RUTA_ERA5_CANARIAS,
    exist_ok=True
)

print("Carpeta Canarias:")
print(RUTA_ERA5_CANARIAS)

print("\nExiste:")
print(os.path.exists(RUTA_ERA5_CANARIAS))

In [ ]:
# ============================================================
# COMPROBAR CONFIGURACIÓN ANTES DE DESCARGAR
# ============================================================

print("Años:")
print(sorted(meses_por_ano.keys()))

print("\nMeses por año:")

for ano, meses in meses_por_ano.items():
    print(
        ano,
        "->",
        meses,
        "| n =", len(meses)
    )

print(
    "\nTotal meses:",
    sum(len(meses) for meses in meses_por_ano.values())
)

In [ ]:
# ============================================================
# DESCARGAR ERA5-LAND CANARIAS
# ============================================================

for ano, meses in meses_por_ano.items():

    archivo_salida = os.path.join(
        RUTA_ERA5_CANARIAS,
        f"ERA5Land_Canarias_mensual_{ano}.nc"
    )

    # Evitar repetir descargas ya realizadas
    if os.path.exists(archivo_salida):
        print(f"{ano}: ya existe -> se omite")
        continue

    print(f"\nDescargando Canarias {ano}...")
    print("Meses:", meses)

    request = {
        "product_type": [
            "monthly_averaged_reanalysis"
        ],

        "variable": variables_era5,

        "year": [
            str(ano)
        ],

        "month": meses,

        "time": [
            "00:00"
        ],

        "data_format": "netcdf",

        "download_format": "unarchived",

        # Norte, Oeste, Sur, Este
        "area": [
            29.5, -18.5,
            27.0, -13.0
        ]
    }

    cds.retrieve(
        "reanalysis-era5-land-monthly-means",
        request,
        archivo_salida
    )

    print(f"{ano}: OK")
    print("Guardado en:")
    print(archivo_salida)

print("\nDescarga de Canarias terminada.")

In [ ]:
import os
import xarray as xr
import pandas as pd

resumen_canarias = []

for ano in range(2016, 2024):

    ruta = os.path.join(
        RUTA_ERA5_CANARIAS,
        f"ERA5Land_Canarias_mensual_{ano}.nc"
    )

    ds_tmp = xr.open_dataset(ruta)

    resumen_canarias.append({
        "ano": ano,
        "n_meses": ds_tmp.sizes["valid_time"],
        "fecha_inicio": pd.to_datetime(
            ds_tmp["valid_time"].values[0]
        ).strftime("%Y-%m"),
        "fecha_fin": pd.to_datetime(
            ds_tmp["valid_time"].values[-1]
        ).strftime("%Y-%m")
    })

    ds_tmp.close()

resumen_canarias_df = pd.DataFrame(resumen_canarias)

display(resumen_canarias_df)

print(
    "\nTotal meses Canarias:",
    resumen_canarias_df["n_meses"].sum()
)

In [ ]:
ruta_canarias_prueba = os.path.join(
    RUTA_ERA5_CANARIAS,
    "ERA5Land_Canarias_mensual_2022.nc"
)

ds_can = xr.open_dataset(ruta_canarias_prueba)

print(ds_can.sizes)

lat_can = ds_can["latitude"].values
lon_can = ds_can["longitude"].values

lon_grid_can, lat_grid_can = np.meshgrid(
    lon_can,
    lat_can
)

puntos_era5_can = pd.DataFrame({
    "longitude": lon_grid_can.ravel(),
    "latitude": lat_grid_can.ravel()
})

puntos_era5_can = gpd.GeoDataFrame(
    puntos_era5_can,
    geometry=gpd.points_from_xy(
        puntos_era5_can["longitude"],
        puntos_era5_can["latitude"]
    ),
    crs="EPSG:4326"
)

print("Puntos ERA5 Canarias:", len(puntos_era5_can))

In [ ]:
provincias_canarias_final = provincias_final[
    provincias_final["provincia_estandar"].isin(
        ["Las Palmas", "Santa Cruz de Tenerife"]
    )
].copy()

puntos_can_provincia = gpd.sjoin(
    puntos_era5_can,
    provincias_canarias_final[
        ["provincia_estandar", "geometry"]
    ],
    how="inner",
    predicate="within"
)

conteo_canarias = (
    puntos_can_provincia
    .groupby("provincia_estandar")
    .size()
    .to_frame("n_puntos_ERA5")
)

display(conteo_canarias)

In [ ]:
# ============================================================
# COMPROBAR COBERTURA DEL ERA5 DEFINITIVO
# ============================================================

ruta_era5_definitivo = os.path.join(
    RUTA_ERA5,
    "ERA5Land_mensual_2022_espana.nc"
)

ds_def = xr.open_dataset(ruta_era5_definitivo)

print("Dimensiones:")
print(ds_def.sizes)

print("\nLatitud:")
print(
    float(ds_def.latitude.min()),
    "a",
    float(ds_def.latitude.max())
)

print("\nLongitud:")
print(
    float(ds_def.longitude.min()),
    "a",
    float(ds_def.longitude.max())
)

print("\nMeses:")
print(ds_def.sizes["valid_time"])

In [ ]:
# ============================================================
# MALLA ERA5 DEFINITIVA ESPAÑA COMPLETA
# ============================================================

latitudes = ds_def["latitude"].values
longitudes = ds_def["longitude"].values

lon_grid, lat_grid = np.meshgrid(
    longitudes,
    latitudes
)

puntos_era5_def = pd.DataFrame({
    "longitude": lon_grid.ravel(),
    "latitude": lat_grid.ravel()
})

puntos_era5_def = gpd.GeoDataFrame(
    puntos_era5_def,
    geometry=gpd.points_from_xy(
        puntos_era5_def["longitude"],
        puntos_era5_def["latitude"]
    ),
    crs="EPSG:4326"
)

puntos_provincia_def = gpd.sjoin(
    puntos_era5_def,
    provincias_final[
        ["provincia_estandar", "geometry"]
    ],
    how="inner",
    predicate="within"
)

conteo_def = (
    puntos_provincia_def
    .groupby("provincia_estandar")
    .size()
    .sort_values()
    .to_frame("n_puntos_ERA5")
)

print(
    "Provincias representadas:",
    puntos_provincia_def["provincia_estandar"].nunique()
)

display(conteo_def)

In [ ]:
# ============================================================
# AGREGAR ERA5-LAND POR PROVINCIA Y MES
# ============================================================

import calendar

# Mapa espacial ERA5 -> provincia
mapa_puntos_provincia = puntos_provincia_def[
    [
        "latitude",
        "longitude",
        "provincia_estandar"
    ]
].copy()

resultados_meteo = []

for ano in range(2016, 2024):

    print(f"Procesando {ano}...")

    ruta = os.path.join(
        RUTA_ERA5,
        f"ERA5Land_mensual_{ano}_espana.nc"
    )

    ds_tmp = xr.open_dataset(ruta)

    # Convertir xarray a tabla
    df_tmp = (
        ds_tmp[
            ["t2m", "d2m", "tp", "ssrd"]
        ]
        .to_dataframe()
        .reset_index()
    )

    # Añadir provincia según la celda ERA5
    df_tmp = df_tmp.merge(
        mapa_puntos_provincia,
        on=["latitude", "longitude"],
        how="inner"
    )

    # Fecha y días del mes
    df_tmp["valid_time"] = pd.to_datetime(
        df_tmp["valid_time"]
    )

    df_tmp["mes"] = df_tmp["valid_time"].dt.month
    df_tmp["ano_calendario"] = df_tmp["valid_time"].dt.year
    df_tmp["dias_mes"] = df_tmp["valid_time"].dt.days_in_month

    # Conversión de unidades
    df_tmp["temp_media_c"] = (
        df_tmp["t2m"] - 273.15
    )

    df_tmp["temp_rocio_c"] = (
        df_tmp["d2m"] - 273.15
    )

    df_tmp["precipitacion_mm"] = (
        df_tmp["tp"]
        * 1000
        * df_tmp["dias_mes"]
    )

    df_tmp["radiacion_mj_m2"] = (
        df_tmp["ssrd"]
        * df_tmp["dias_mes"]
        / 1_000_000
    )

    # Media espacial provincial
    provincial_mes = (
        df_tmp
        .groupby(
            [
                "provincia_estandar",
                "ano_calendario",
                "mes"
            ],
            as_index=False
        )
        .agg(
            temp_media_c=("temp_media_c", "mean"),
            temp_rocio_c=("temp_rocio_c", "mean"),
            precipitacion_mm=("precipitacion_mm", "mean"),
            radiacion_mj_m2=("radiacion_mj_m2", "mean")
        )
    )

    resultados_meteo.append(provincial_mes)

    ds_tmp.close()

# Unir todos los años
meteo_provincial_mensual = pd.concat(
    resultados_meteo,
    ignore_index=True
)

print("\nDimensiones:")
print(meteo_provincial_mensual.shape)

display(meteo_provincial_mensual.head())

In [ ]:
# ============================================================
# ASIGNAR AÑO DE CAMPAÑA / COSECHA
# ============================================================

meteo_provincial_mensual["ano_campana"] = np.where(
    meteo_provincial_mensual["mes"] >= 9,
    meteo_provincial_mensual["ano_calendario"] + 1,
    meteo_provincial_mensual["ano_calendario"]
)

# Solo campañas que existen en nuestro target
meteo_provincial_mensual = (
    meteo_provincial_mensual[
        meteo_provincial_mensual["ano_campana"]
        .between(2017, 2023)
    ]
    .copy()
)

print("Filas:", len(meteo_provincial_mensual))

print("\nMeses por provincia y campaña:")
control_meses = (
    meteo_provincial_mensual
    .groupby(
        ["provincia_estandar", "ano_campana"]
    )
    .size()
)

print(control_meses.value_counts().sort_index())

In [ ]:
# ============================================================
# GUARDAR ERA5-LAND PROVINCIAL MENSUAL
# ============================================================

ruta_meteo_mensual = os.path.join(
    RUTA_PROCESADOS,
    "ERA5Land_provincial_mensual_2017_2023.csv"
)

meteo_provincial_mensual.to_csv(
    ruta_meteo_mensual,
    index=False,
    encoding="utf-8-sig"
)

print("Guardado en:")
print(ruta_meteo_mensual)

print("\nExiste:", os.path.exists(ruta_meteo_mensual))
print("Dimensiones:", meteo_provincial_mensual.shape)

In [ ]:
# ============================================================
# DEFINIR VENTANAS TEMPORALES
# ============================================================

def asignar_ventana(mes):

    if mes in [9, 10, 11, 12]:
        return "sep_dic"

    elif mes in [1, 2]:
        return "ene_feb"

    elif mes in [3, 4]:
        return "mar_abr"

    elif mes in [5, 6]:
        return "may_jun"

    elif mes in [7, 8]:
        return "jul_ago"

    else:
        return None


meteo_provincial_mensual["ventana"] = (
    meteo_provincial_mensual["mes"]
    .apply(asignar_ventana)
)

print(
    meteo_provincial_mensual["ventana"]
    .value_counts()
)

In [ ]:
# ============================================================
# VARIABLES ERA5 POR VENTANA
# ============================================================

meteo_ventanas = (
    meteo_provincial_mensual
    .groupby(
        [
            "provincia_estandar",
            "ano_campana",
            "ventana"
        ],
        as_index=False
    )
    .agg(
        temp_media_c=("temp_media_c", "mean"),
        temp_rocio_c=("temp_rocio_c", "mean"),
        precipitacion_mm=("precipitacion_mm", "sum"),
        radiacion_mj_m2=("radiacion_mj_m2", "sum")
    )
)

print("Dimensiones:")
print(meteo_ventanas.shape)

display(meteo_ventanas.head(10))

In [ ]:
# ============================================================
# FORMATO ANCHO PARA EL MODELO
# ============================================================

meteo_features = meteo_ventanas.pivot(
    index=[
        "provincia_estandar",
        "ano_campana"
    ],
    columns="ventana",
    values=[
        "temp_media_c",
        "temp_rocio_c",
        "precipitacion_mm",
        "radiacion_mj_m2"
    ]
)

# Convertir MultiIndex de columnas a nombres simples
meteo_features.columns = [
    f"{variable}_{ventana}"
    for variable, ventana in meteo_features.columns
]

meteo_features = (
    meteo_features
    .reset_index()
)

print("Dimensiones:")
print(meteo_features.shape)

print("\nColumnas:")
print(meteo_features.columns.tolist())

display(meteo_features.head())

In [ ]:
# ============================================================
# VARIABLES DE CAMPAÑA COMPLETA
# ============================================================

meteo_campana = (
    meteo_provincial_mensual
    .groupby(
        [
            "provincia_estandar",
            "ano_campana"
        ],
        as_index=False
    )
    .agg(
        temp_media_c_campana=("temp_media_c", "mean"),
        temp_rocio_c_campana=("temp_rocio_c", "mean"),
        precipitacion_mm_campana=("precipitacion_mm", "sum"),
        radiacion_mj_m2_campana=("radiacion_mj_m2", "sum")
    )
)

meteo_features = meteo_features.merge(
    meteo_campana,
    on=[
        "provincia_estandar",
        "ano_campana"
    ],
    how="left"
)

print("Dimensiones finales ERA5:")
print(meteo_features.shape)

print("\nNúmero de variables:")
print(len(meteo_features.columns))

display(meteo_features.head())

In [ ]:
# ============================================================
# CONTROL FINAL ERA5
# ============================================================

print("Filas:", len(meteo_features))

print(
    "Combinaciones provincia-campaña duplicadas:",
    meteo_features.duplicated(
        subset=[
            "provincia_estandar",
            "ano_campana"
        ]
    ).sum()
)

print(
    "Valores nulos totales:",
    meteo_features.isna().sum().sum()
)

print("\nCampañas:")
print(
    sorted(
        meteo_features["ano_campana"].unique()
    )
)

In [ ]:
# ============================================================
# GUARDAR FEATURES ERA5-LAND
# ============================================================

ruta_features_era5 = os.path.join(
    RUTA_PROCESADOS,
    "ERA5Land_features_provincia_campana_2017_2023.csv"
)

meteo_features.to_csv(
    ruta_features_era5,
    index=False,
    encoding="utf-8-sig"
)

print("Guardado:")
print(ruta_features_era5)

print("Existe:", os.path.exists(ruta_features_era5))

# Fusionar MAPA y ERA-LAND

In [ ]:
# ============================================================
# FUSIONAR MAPA + ERA5-LAND
# ============================================================

meteo_para_modelo = meteo_features.rename(
    columns={
        "ano_campana": "ano"
    }
)

dataset_mapa_era5 = target.merge(
    meteo_para_modelo,
    on=[
        "provincia_estandar",
        "ano"
    ],
    how="left",
    validate="many_to_one"
)

print("Filas target original:", len(target))
print("Filas después del merge:", len(dataset_mapa_era5))

print(
    "Observaciones sin meteorología:",
    dataset_mapa_era5[
        "temp_media_c_campana"
    ].isna().sum()
)

display(dataset_mapa_era5.head())

In [ ]:
# ============================================================
# GUARDAR DATASET MAPA + ERA5
# ============================================================

ruta_dataset_mapa_era5 = os.path.join(
    RUTA_PROCESADOS,
    "dataset_MAPA_ERA5_trigo_cebada_2017_2023.csv"
)

dataset_mapa_era5.to_csv(
    ruta_dataset_mapa_era5,
    index=False,
    encoding="utf-8-sig"
)

print("Guardado permanentemente en Drive:")
print(ruta_dataset_mapa_era5)

print("\nDimensiones:")
print(dataset_mapa_era5.shape)